In [ ]:
/**
 * ============================================================================
 *  IP ADDRESSES - Complete Simple Notes (Hinglish)
 *  Beginners ke liye pura detail mein samjhein
 * ============================================================================
 *
 * ============================================================================
 *  INTRO: IP Address Kya Hai Aur Kyun Zaroori Hai?
 * ============================================================================
 *
 * Routers, firewalls, load balancers, cloud networks, aur DNS - sabko ek cheez
 * chahiye packet move karne se pehle: "Kahan bhejna hai?" (somewhere to send it)
 *
 * - IP ADDRESS wahi "somewhere" hai
 * - Yeh ek NETWORK ADDRESS hai jo decide karta hai packets kahan jaayenge
 *
 * IMPORTANT - IP Address permanent identity NAHI hai:
 *   - Yeh ek NETWORK INTERFACE ko assign hota hai
 *   - Jaise: laptop ka WiFi adapter, server ka network card, VM, ya container
 *   - Ek laptop ke paas EK SE ZYADA IP addresses ho sakte hain ek saath
 *   - Container har baar start hone pe NAYA IP le sakta hai
 *
 * NAT ke wajah se HAZAARON users EK public IPv4 address share kar sakte hain
 * - Ek single public IP load balancer, CDN edge, ya anycast service bhi ho
 *   sakta hai (specific server nahi)
 *
 * SYSTEM DESIGN mein IP addresses ko ROUTING COORDINATES samjho:
 *   - Yeh user identity NAHI hai
 *   - Yeh device identity NAHI hai
 *   - Yeh proof of trust NAHI hai
 *   - IP address ka matlab hai "yahan bhejo", bas itna hi
 *
 * ============================================================================
 *  SECTION 1: WHY IP ADDRESSING MATTERS
 * ============================================================================
 *
 * IP addressing bohot cheezon pe asar karta hai:
 *   - VPC aur subnet design (Cloud network ka layout)
 *   - Kubernetes pod aur service networking
 *   - Load balancer listeners aur target groups
 *   - Firewall rules aur security groups
 *   - Database allowlists (kaun connect kar sakta hai)
 *   - DNS records
 *   - NAT gateways aur egress controls (bahar jaane ka raasta)
 *   - Private service connectivity
 *
 * Agar ADDRESSING PLAN galat hai toh:
 *   - Systems connect hone mein mushkil hoti hai
 *   - Secure karna mushkil hota hai
 *   - Baad mein merge karna bahut painful hota hai
 *
 * ============================================================================
 *  SECTION 2: IPv4 (Purana Wala IP - Abhi bhi Sabse Zyada Use Hota Hai)
 * ============================================================================
 *
 * - IPv4 IP ka OLDER version hai, lekin abhi bhi BAHUT zyada use hota hai
 * - Yeh 32-BIT addresses use karta hai
 * - Format: chaar numbers dots se alag (jaise 192.168.1.25)
 *
 * Binary example:
 *   192        . 168       . 1         . 25
 *   11000000     10101000    00000001    00011001
 *   (8 bits)     (8 bits)    (8 bits)    (8 bits)
 *
 * Har number ko OCTET kehte hain
 *   - Har octet ka range: 0 se 255
 *   - 32-bit space = 2^32 = lagbhag 4.3 BILLION possible addresses
 *   - Jab IPv4 banaya gaya tha toh yeh BAHUT zyada lagta tha
 *   - Lekin aaj: phones, laptops, cloud instances, containers, connected
 *     devices, aur global services ke liye yeh CHHOTA hai
 *
 * Public IPv4 Pool EXHAUSTED hai:
 *   - IANA ne 2011 mein apne LAST LARGE BLOCKS Regional Internet Registries
 *     ko de diye
 *   - Regional pools bhi dheere dheere kam ho gaye
 *   - IPv4 abhi bhi chalta hai kyunki industry ne isko stretch kiya:
 *     * Private addressing
 *     * NAT
 *     * Address markets
 *     * Careful allocation
 *
 * ============================================================================
 *  HISTORICAL ADDRESS CLASSES (Classful Addressing - Purana Tarika)
 * ============================================================================
 *
 * Purane IPv4 mein CLASSFUL ADDRESSING use hoti thi
 * - Address ke pehle bits decide karte the ki network kitna bada hai
 *
 *   ┌──────────┬─────────────────────┬────────────────────┬──────────────────────────┐
 *   │ Class    │ First Octet Range   │ Default Mask       │ Historical Use           │
 *   ├──────────┼─────────────────────┼────────────────────┼──────────────────────────┤
 *   │ A        │ 1-126               │ 255.0.0.0 (/8)     │ Very large networks      │
 *   ├──────────┼─────────────────────┼────────────────────┼──────────────────────────┤
 *   │ B        │ 128-191             │ 255.255.0.0 (/16)  │ Medium networks          │
 *   ├──────────┼─────────────────────┼────────────────────┼──────────────────────────┤
 *   │ C        │ 192-223             │ 255.255.255.0(/24) │ Small networks           │
 *   ├──────────┼─────────────────────┼────────────────────┼──────────────────────────┤
 *   │ D        │ 224-239             │ N/A                │ Multicast                │
 *   ├──────────┼─────────────────────┼────────────────────┼──────────────────────────┤
 *   │ E        │ 240-255             │ N/A                │ Reserved or experimental │
 *   └──────────┴─────────────────────┴────────────────────┴──────────────────────────┘
 *
 * PROBLEM with classful addressing:
 *   - Bahut saara address space WASTE hota tha
 *   - Example: Agar company ko 500 addresses chahiye, toh Class C (254 usable
 *     hosts) kam padta hai, toh usse bahut bada Class B allocation milta tha
 *   - Yeh model SCALE nahi karta tha
 *
 * Modern networks CIDR use karte hain, CLASSFUL ADDRESSING nahi
 * - Aaj bhi log casually "Class A" ya "Class C" bolte hain
 * - Lekin production designs mein CIDR notation use karna chahiye
 *
 * ============================================================================
 *  SECTION 3: CIDR AND SUBNETS (Aadhaar Samjho)
 * ============================================================================
 *
 * CIDR ka full form: "Classless Inter-Domain Routing"
 * - Yeh IP address block describe karne ka COMMON tarika hai
 *
 * Slashes ke baad ka number: PREFIX LENGTH kehte hain
 *   - Yeh batata hai kitne leading bits NETWORK identify karte hain
 *
 * EXAMPLE: 192.168.1.0/24
 *   - Pehle 24 bits NETWORK identify karte hain
 *   - Baaki 8 bits network ke andar addresses ke liye hain
 *   - Block mein: 2^8 = 256 IPv4 addresses
 *
 * IPv4 Address Structure:
 *   Network Portion (24 bits) | Host Portion (8 bits)
 *   192 . 168 . 1 . 0-255  =  192.168.1.0/24  (256 IPv4 addresses)
 *
 * TRADITIONAL IPv4 SUBNET ke Rules:
 *   - Pehla address = NETWORK ADDRESS
 *   - Aakhri address = BROADCAST ADDRESS
 *   - Beech ke: 254 USABLE HOST ADDRESSES (/24 mein)
 *
 * EXCEPTIONS (Alag cases):
 *   - /32: Ek SINGLE IPv4 address identify karta hai (exact routes ya
 *     firewall rules ke liye)
 *   - /31: Point-to-point links ke liye use ho sakta hai (broadcast address
 *     ki zaroorat nahi)
 *   - Cloud providers aksar subnet ke andar addresses RESERVE karte hain
 *     (AWS, Azure, GCP mein jo paper pe usable lagta hai woh sab nahi milta)
 *   - IPv6 mein broadcast NAHI hai, toh IPv4 ka "minus two" rule waise ka
 *     waise apply nahi hota
 *
 * ============================================================================
 *  COMMON IPv4 CIDR BLOCKS (Yaad Rakho)
 * ============================================================================
 *
 *   ┌──────────┬────────────────────┬──────────────────┬───────────────────────────────────┐
 *   │ CIDR     │ Subnet Mask        │ Total Addresses  │ Typical Use                       │
 *   ├──────────┼────────────────────┼──────────────────┼───────────────────────────────────┤
 *   │ /8       │ 255.0.0.0          │ 16,777,216       │ Very large private networks,      │
 *   │          │                    │                  │ large allocations                 │
 *   ├──────────┼────────────────────┼──────────────────┼───────────────────────────────────┤
 *   │ /16      │ 255.255.0.0        │ 65,536           │ VPCs, corporate networks,         │
 *   │          │                    │                  │ large environments                │
 *   ├──────────┼────────────────────┼──────────────────┼───────────────────────────────────┤
 *   │ /20      │ 255.255.240.0      │ 4,096            │ Medium subnets or environment     │
 *   │          │                    │                  │ ranges                            │
 *   ├──────────┼────────────────────┼──────────────────┼───────────────────────────────────┤
 *   │ /28      │ 255.255.255.240    │ 16               │ Small network segments            │
 *   ├──────────┼────────────────────┼──────────────────┼───────────────────────────────────┤
 *   │ /32      │ 255.255.255.255    │ 1                │ Single host route or exact        │
 *   │          │                    │                  │ firewall match                    │
 *   └──────────┴────────────────────┴──────────────────┴───────────────────────────────────┘
 *
 * EXAMPLE: 10.0.0.0/20 ka Calculation:
 *   - Prefix /20 = 12 host bits remaining
 *   - Address count: 2^12 = 4,096
 *   - Range: 10.0.0.0 se 10.0.15.255 tak
 *   - Traditional usable host range: 10.0.0.1 se 10.0.15.254 tak
 *
 * Yeh calculation IMPORTANT hai jab aap subnets design karte hain:
 *   - Application tiers ke liye
 *   - Kubernetes clusters ke liye
 *   - NAT gateways ke liye
 *   - Managed databases ke liye
 *
 * ============================================================================
 *  SECTION 4: DESIGNING SUBNETS IN CLOUD SYSTEMS
 * ============================================================================
 *
 * Cloud platforms IP planning ko EASY dikhate hain:
 *   - Aap VPC ke liye ek CIDR block choose karte hain
 *   - Phir usse subnets mein split karte hain
 *
 * ASLI MUSHKIL BAAD MEIN AATI HAI jab:
 *   - Networks connect hone hain
 *   - Merge hone hain
 *   - Expand hone hain
 *   - Company ke apne data centers tak pahunchna hai
 *
 * COMMON STARTING POINT (Ek Cloud Network ka Layout):
 *   ┌────────────────────┬──────────────────────────────────────────┐
 *   │ Range              │ Purpose                                  │
 *   ├────────────────────┼──────────────────────────────────────────┤
 *   │ 10.0.0.0/16        │ VPC ya virtual network                   │
 *   ├────────────────────┼──────────────────────────────────────────┤
 *   │ 10.0.0.0/20        │ Public subnets across availability zones │
 *   ├────────────────────┼──────────────────────────────────────────┤
 *   │ 10.0.16.0/20       │ Private application subnets              │
 *   ├────────────────────┼──────────────────────────────────────────┤
 *   │ 10.0.32.0/20       │ Data subnets                             │
 *   ├────────────────────┼──────────────────────────────────────────┤
 *   │ 10.0.48.0/20       │ Kubernetes nodes ya pods (model pe        │
 *   │                    │ depend karta hai)                         │
 *   └────────────────────┴──────────────────────────────────────────┘
 *
 * GOOD IP PLANS kya karte hain:
 *   - Growth ke liye JAGAH chhodte hain
 *   - Baad mein connect hone wale networks se OVERLAP avoid karte hain
 *
 * OVERLAPPING CIDR RANGES = SABSE MEHENGA CLOUD NETWORKING MISTAKE:
 *   - Agar do VPCs dono 10.0.0.0/16 use karte hain
 *   - Toh unke beech routing mushkil ya IMPOSSIBLE ho sakta hai
 *   - Bina NAT, proxying, renumbering, ya complex translation ke kaam nahi
 *     chalega
 *
 * PRACTICAL GUIDANCE (Amal Mein Lao):
 *   1. Har jagah SAME DEFAULT RANGE use mat karo
 *   2. Future environments aur regions ke liye SPACE RESERVE karo
 *   3. Production, staging, data, aur shared services ko PREDICTABLE ranges
 *      mein rakho
 *   4. Cloud CIDRs choose karne se pehle existing CORPORATE, VPN, PARTNER, aur
 *      DATA-CENTER ranges CHECK karo
 *   5. Kubernetes pod IP usage EARLY mein account karo - bohot pods wale
 *      clusters addresses EXPECTED SE TEZ consume karte hain
 *   6. Yaad rakho cloud subnets PROVIDER-SPECIFIC addresses reserve kar sakte
 *      hain
 *
 * ADDRESS PLAN = INFRASTRUCTURE ARCHITECTURE
 *   - Isse treat waise hi karo jaise architecture ko karte ho
 *
 * WATCH FOR (In Cheezon Ka Dhyan Rakho):
 *   - Too-small VPC ranges
 *   - Overlapping networks after mergers ya cloud migrations
 *   - Kubernetes pod address exhaustion (IPs khatam hona)
 *   - NAT port exhaustion
 *   - IPv4-only dependencies in systems jo dono IPv4 aur IPv6 support karte hain
 *   - Hidden assumptions in firewall rules aur allowlists
 *
 * JITNI JALDI ADDRESSING DESIGN FIX KAROGE, UTNA SASTA HOGA
 *
 * ============================================================================
 *  PREFER NAMES FOR SERVICE CONTRACTS
 * ============================================================================
 *
 * - Applications ko usually DNS names ya service discovery names pe depend
 *   karna chahiye
 *   - Hard-coded IP addresses NAHI use karna chahiye
 *   - IPs IMPLEMENTATION DETAILS hain (internal cheez hain)
 *
 * NAMES se kya fayda:
 *   - Failover (backup server pe switch karna)
 *   - Migration (server move karna)
 *   - Load balancing
 *   - Certificate validation (SSL certificates)
 *   - Regional routing
 *
 * Hard-coded IPs kabhi kabhi appropriate hain INFRASTRUCTURE CONTROLS ke liye,
 * lekin woh RARE hone chahiye aur DOCUMENTED hone chahiye
 *
 * ============================================================================
 *  SUMMARY (Section 1-4 ka Saransh)
 * ============================================================================
 *
 * - IP addresses ROUTING ADDRESSES hain
 *   - Yeh batate hain ki packets network context ke andar kahan jaayein
 *   - Yeh PROVE NAHI karte ki user kaun hai ya device permanently kya hai
 *
 * - IPv4 abhi bhi BAHUT zyada use hota hai chahe public IPv4 space exhausted hai
 * - CIDR ne CLASSFUL ADDRESSING ko replace kiya as the basis of modern subnet
 *   design
 *
 * ============================================================================
 *  SECTION 5: PUBLIC AND PRIVATE IP ADDRESSES
 * ============================================================================
 *
 * IPv4 mein sabse important FARK yeh hai ki address PUBLIC INTERNET pe route
 * ho sakta hai ya nahi
 *
 * ============================================================================
 *  PUBLIC IP ADDRESSES (Bahar Se Pahunch Sakte Hain)
 * ============================================================================
 *
 * - Public IP address INTERNET ke through reach kiya ja sakta hai
 * - Public addresses allocate hote hain through:
 *     * Regional Internet Registries (RIRs)
 *     * ISPs
 *     * Cloud providers
 *     * Network operators
 *   - Public address space COORDINATED hai taaki alag networks GLOBALLY ROUTED
 *     ranges accidentally same use na karein
 *
 * PUBLIC IPs KAHAN USE HOTE HAIN:
 *   1. Internet-facing load balancers
 *   2. CDN aur edge services
 *   3. NAT gateways
 *   4. VPN endpoints
 *   5. Public DNS targets
 *   6. Mail servers aur other externally reachable services
 *
 * IMPORTANT - "PUBLIC" ka matlab "SAFE" NAHI hai:
 *   - Public IP = REACHABLE (bas itna hi)
 *   - Security abhi bhi depend karti hai:
 *     * Firewall rules
 *     * Authentication
 *     * Patching
 *     * DDoS protection
 *     * Application controls
 *
 * ============================================================================
 *  PRIVATE IPv4 ADDRESSES (Andar Ka Network)
 * ============================================================================
 *
 * - Private IPv4 ranges RFC 1918 se defined hain
 * - KOI BHI private network inhe REUSE kar sakta hai
 *   - Kyunki public internet ke routers inhe ROUTE NAHI karte
 *
 *   ┌────────────────────────────────┬────────────────────┬──────────────┬──────────────────────────────┐
 *   │ Range                          │ CIDR               │ Address Count│ Common Use                   │
 *   ├────────────────────────────────┼────────────────────┼──────────────┼──────────────────────────────┤
 *   │ 10.0.0.0 to 10.255.255.255     │ 10.0.0.0/8         │ 16,777,216   │ Cloud VPCs, large corporate  │
 *   │                                │                    │              │ networks                     │
 *   ├────────────────────────────────┼────────────────────┼──────────────┼──────────────────────────────┤
 *   │ 172.16.0.0 to 172.31.255.255   │ 172.16.0.0/12      │ 1,048,576    │ Enterprise networks,         │
 *   │                                │                    │              │ containers, internal         │
 *   │                                │                    │              │ platforms                    │
 *   ├────────────────────────────────┼────────────────────┼──────────────┼──────────────────────────────┤
 *   │ 192.168.0.0 to 192.168.255.255 │ 192.168.0.0/16     │ 65,536       │ Home networks, small         │
 *   │                                │                    │              │ offices, labs                │
 *   └────────────────────────────────┴────────────────────┴──────────────┴──────────────────────────────┘
 *
 * Private addresses ko sirf US NETWORK ke andar UNIQUE hona chahiye:
 *   - Aapka laptop aur padosi ka laptop DONO 192.168.1.10 ho sakte hain
 *   - Kyunki un addresses alag routers ke peeche (behind) hain
 *
 * ============================================================================
 *  CARRIER-GRADE NAT (ISP Level NAT)
 * ============================================================================
 *
 * Ek aur range ISP aur mobile networks mein aksar dikhti hai:
 *   ┌────────────────────────────────┬────────────────────┬─────────────────────────────────────┐
 *   │ Range                          │ CIDR               │ Purpose                             │
 *   ├────────────────────────────────┼────────────────────┼─────────────────────────────────────┤
 *   │ 100.64.0.0 to 100.127.255.255  │ 100.64.0.0/10      │ Shared address space for            │
 *   │                                │                    │ carrier-grade NAT                   │
 *   └────────────────────────────────┴────────────────────┴─────────────────────────────────────┘
 *
 * CARRIER-GRADE NAT kya karta hai:
 *   - Providers ko allow karta hai ki bahut saare customers ko CHHOTE number of
 *     public IPv4 addresses ke peeche rakh dein
 *   - IPv4 scarcity mein help karta hai
 *   - Lekin YEH PROBLEMS bhi laata hai:
 *     * Inbound connectivity mushkil hoti hai
 *     * Abuse tracking mushkil hoti hai
 *     * Kuch peer-to-peer protocols mushkil ho jaate hain
 *
 * ============================================================================
 *  SECTION 6: NAT (Network Address Translation)
 * ============================================================================
 *
 * NAT ka full form: "Network Address Translation"
 * - NAT packet addresses ko REWRITE karta hai jab traffic ek NETWORK BOUNDARY
 *   cross karta hai
 * - Sabse common form: SOURCE NAT for OUTBOUND traffic
 *
 * NAT KAISE KAAM KARTA HAI (Step by step):
 *   1. PRIVATE host ek packet internet pe bhejta hai
 *   2. NAT device PRIVATE SOURCE IP ko PUBLIC IP se REPLACE karta hai
 *   3. NAT device woh MAPPING record karta hai
 *   4. Jab RESPONSE wapas aata hai, NAT device us MAPPING ko use karta hai
 *   5. Response SAHI INTERNAL HOST ko bhejta hai
 *
 * NAT ka Diagram:
 *   Private Network:
 *     Phone: 192.168.1.11
 *     Laptop: 192.168.1.10
 *     TV: 192.168.1.12
 *        |
 *        v
 *   NAT Router:
 *     Private: 192.168.1.10
 *     Public: 203.0.113.5 (example address)
 *        |
 *        v
 *   Outbound traffic appears from 203.0.113.5
 *        |
 *        v
 *   Internet
 *        |
 *        v
 *   Service: 198.51.100.20 (example address)
 *
 * NAT ek wajah hai ki IPv4 ADDRESS EXHUSTION survive kar paya
 * Lekin yeh OPERATIONAL COMPLEXITY bhi laata hai:
 *
 *   1. INBOUND CONNECTIONS ke liye chahiye:
 *      - Port forwarding, load balancers, VPNs, ya proxies
 *
 *   2. LOGS mein original client IP preserve karna padta hai:
 *      - Headers ya proxy protocol ke through
 *      - Jab traffic proxies ya load balancers se guzarta hai
 *
 *   3. NAT TABLES FULL ho sakti hain:
 *      - High connection churn ke under
 *
 *   4. LONG-LIVED IDLE CONNECTIONS:
 *      - NAT timeouts se DROP ho sakti hain
 *
 *   5. MULTIPLE CLIENTS ek public IP share kar sakte hain:
 *      - Isliye IP-based rate limiting UNRELATED USERS ko punish kar sakta hai
 *
 *   6. PROTOCOLS jo IP addresses PAYLOAD mein embed karte hain:
 *      - Unhe SPECIAL HANDLING chahiye ho sakti hai
 *
 * Cloud systems mein NAT GATEWAYS bahut common hain:
 *   - Private subnets ke liye jo outbound internet access chahte hain
 *   - Yeh USEFUL hain lekin yeh CAPACITY, COST, aur AVAILABILITY DEPENDENCIES
 *     bhi ban jaate hain
 *
 * HIGH-THROUGHPUT WORKLOADS jo NAT gateways ko stress kar sakte hain:
 *   - Package downloads
 *   - Telemetry agents
 *   - Model-serving nodes jo large files pull karte hain
 *   - Agar limits planned nahi hain toh problem hogi
 *
 * NAT IPv4 scarcity ka WORKAROUND hai
 * - Yeh CLEAN ADDRESSING MODEL ka SUBSTITUTE NAHI hai
 *
 * ============================================================================
 *  SECTION 7: IPv6 (Naya Wala IP - Bada Address Space)
 * ============================================================================
 *
 * - IPv6 128-BIT addresses use karta hai (IPv4 se bahut bada!)
 * - Isse BAHUT ZYADA ADDRESS SPACE milta hai
 * - IPv4 jaise addresses conserve karne ki zaroorat NAHI hai
 *
 * IPv6 address ka FORMAT: 8 groups of hexadecimal digits
 *   Example: 2001:0db8:85a3:0000:0000:8a2e:0370:7334
 *
 * IPv6 addresses DARAWNAY lag sakte hain, lekin sirf 2 SIMPLE SHORTENING
 * RULES hain:
 *
 *   Rule 1: GROUP ke shuru ke ZEROS hatao
 *     - 0db8 ban jaata hai db8
 *
 *   Rule 2: EK consecutive run of ZERO GROUPS ko :: se replace karo
 *     - 0000:0000 ko :: se replace karo
 *
 * Upar wala address aise likha ja sakta hai:
 *   2001:db8:85a3::8a2e:370:7334
 *
 * DOCUMENTATION PREFIXES:
 *   - 2001:db8::/32 - reserved for IPv6 documentation examples
 *   - 192.0.2.0/24, 198.51.100.0/24, 203.0.113.0/24 - reserved for IPv4 examples
 *
 * ============================================================================
 *  WHAT CHANGES WITH IPv6 (IPv6 Mein Kya Badalta Hai)
 * ============================================================================
 *
 * IPv6 sirf ADDRESS LENGTH nahi badalta - bahut kuch alag hai:
 *
 *   1. IPv6 mein BROADCAST NAHI hai
 *      - Yeh MULTICAST aur NEIGHBOR DISCOVERY use karta hai
 *
 *   2. IPv6 subnets commonly /64 hote hain
 *      - Especially LAN-style networks ke liye
 *
 *   3. IPv6 routers packets ko transit mein SPLIT (fragment) NAHI karte
 *      - Hosts ko khud discover karna chahiye ki path kitna bada packet handle
 *        kar sakta hai
 *
 *   4. LINK-LOCAL ADDRESSES: fe80::/10
 *      - Yeh IPv6 networks mein NORMAL hain
 *      - Local-link IPv6 communication ke liye REQUIRED hain
 *
 *   5. ADDRESS ASSIGNMENT ke tarike:
 *      - SLAAC (Stateless Address Autoconfiguration) - IPv6 ka built-in method
 *        jo hosts ko router announcements se apne addresses banane deta hai
 *      - DHCPv6
 *      - Static configuration
 *      - Cloud-provider mechanisms
 *
 *   6. IPv6 mein addresses CONSERVE karne ke liye NAT ki zaroorat NAHI hai
 *      - Lekin IPv6 FIREWALLING abhi bhi ESSENTIAL hai
 *
 * ============================================================================
 *  IPv6 ADOPTION STATUS
 * ============================================================================
 *
 * - IPv6 adoption ab THEORETICAL nahi hai - yeh REAL mein use hota hai
 * - Common in:
 *     * Mobile networks
 *     * Large ISPs
 *     * Consumer broadband
 *     * Content networks
 *     * Major cloud platforms
 *
 * - Lekin adoption UNEVEN hai:
 *     * Country ke hisaab se
 *     * Provider ke hisaab se
 *     * Enterprise network ke hisaab se
 *     * Cloud architecture ke hisaab se
 *
 * - Isliye MOST PRODUCTION SYSTEMS ko DONO handle karna padta hai:
 *     IPv4 aur IPv6 dono
 *
 * DUAL-STACK (Usual Transition Model):
 *   - Matlab service DONO support karta hai: IPv4 AUR IPv6 ek saath
 *   - Yeh PUBLISH karta hai:
 *     * A records = IPv4 ke liye
 *     * AAAA records = IPv6 ke liye
 *   - Clients pehle IPv6 try karte hain (agar available hai)
 *   - Phir IPv4 pe FALLBACK karte hain (zaroorat padne pe)
 *
 * DUAL-STACK ke PROS aur CONS:
 *   + Hard cutover se bachata hai (ek din sab switch nahi karna padta)
 *   - Lekin yeh FAIL HO SAKNE KE PATHS DOUBLE kar deta hai
 *   - Dono ke liye handle karna padta hai:
 *     * Firewalls
 *     * DNS
 *     * Load balancers
 *     * Service discovery
 *     * Metrics
 *     * Logs
 *     * Incident runbooks
 *
 * ============================================================================
 *  SECTION 8: SPECIAL IP ADDRESSES (Khaas Addresses Jo Bachaate Hain Time)
 * ============================================================================
 *
 * Kuch address ranges SPECIAL BEHAVIOUR rakhte hain
 * - Inhe jaanna debugging mein TIME BACHATA hai
 *
 *   ┌─────────────────────┬──────────────────────────┬──────────────────────────────────────────────────┐
 *   │ Address or Range    │ Name                     │ Purpose                                          │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 127.0.0.0/8         │ IPv4 loopback            │ Traffic same host pe wapas aata hai; 127.0.0.1   │
 *   │                     │                          │ sabse familiar address hai                       │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ ::1/128             │ IPv6 loopback            │ IPv6 loopback address                            │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 0.0.0.0             │ Unspecified address      │ Bind address meaning ALL IPv4 interfaces, ya     │
 *   │                     │                          │ placeholder source before assignment             │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ ::                  │ IPv6 unspecified address │ IPv6 equivalent of unspecified address           │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 0.0.0.0/0           │ Default IPv4 route       │ Agar koi specific route nahi hai toh KISI BHI    │
 *   │                     │                          │ IPv4 destination ko match karta hai              │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ ::/0                │ Default IPv6 route       │ Agar koi specific route nahi hai toh KISI BHI    │
 *   │                     │                          │ IPv6 destination ko match karta hai              │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 255.255.255.255     │ Limited broadcast        │ Local IPv4 network pe broadcast; DHCP discovery  │
 *   │                     │                          │ mein common                                     │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 169.254.0.0/16      │ IPv4 link-local          │ DHCP fail hone pe self-assigned; cloud metadata  │
 *   │                     │                          │ patterns bhi yahan hain                          │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ fe80::/10           │ IPv6 link-local          │ Local-link IPv6 communication ke liye required   │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 224.0.0.0/4         │ IPv4 multicast           │ Multicast groups ke liye one-to-many delivery    │
 *   ├─────────────────────┼──────────────────────────┼──────────────────────────────────────────────────┤
 *   │ 100.64.0.0/10       │ Shared address space     │ Carrier-grade NAT                                │
 *   └─────────────────────┴──────────────────────────┴──────────────────────────────────────────────────┘
 *
 * ============================================================================
 *  LOOPBACK (Apne Aap Se Baat Karna)
 * ============================================================================
 *
 * Jab aap 127.0.0.1 ya localhost se connect karte hain:
 *   - Traffic LOCAL MACHINE pe hi rehta hai
 *   - Yeh local network stack se guzarta hai
 *   - Lekin PHYSICAL NETWORK tak KABHI NAHI pahunchta
 *
 * LOOPBACK useful hai:
 *   - Development ke liye (code test karna)
 *   - Health checks (service chal rahi hai ya nahi)
 *   - Sidecars (proxy containers)
 *   - Local agents
 *   - Local processes ke beech communication
 *
 * BIND ADDRESSES ka DHYAN RAKHO (Bahut Important!):
 *   - 127.0.0.1 pe bind karna = Service sirf LOCAL MACHINE pe visible hai
 *   - 0.0.0.0 pe bind karna = Service HOST FIREWALL aur network path ke
 *     allow karne pe SABHI IPv4 interfaces pe listen karta hai
 *   - Yeh FARK bahut saari problems create karta hai:
 *     "WORKS LOCALLY BUT NOT FROM ANOTHER MACHINE" (yeh woh problem hai!)
 *
 * ============================================================================
 *  0.0.0.0 KA MATLAB (Context Pe Depend Karta Hai)
 * ============================================================================
 *
 * 0.0.0.0 alag alag context mein ALAG matlab rakhta hai:
 *
 *   1. SERVER BIND ADDRESS mein:
 *      - 0.0.0.0:8080 ka matlab hai SABHI IPv4 interfaces pe listen karo
 *
 *   2. ROUTE TABLE mein:
 *      - 0.0.0.0/0 ka matlab hai DEFAULT IPv4 ROUTE
 *
 *   3. SOURCE ADDRESS ke roop mein:
 *      - Yeh bata sakta hai ki host ke paas abhi tak ASSIGNED ADDRESS nahi hai
 *
 * ============================================================================
 *  LINK-LOCAL AND METADATA ENDPOINTS
 * ============================================================================
 *
 * LINK-LOCAL (169.254.x.x):
 *   - Agar laptop ko address 169.254.x.x milta hai
 *   - Toh DHCP likely FAILED hai
 *   - Ya network path to DHCP BROKEN hai
 *
 * CLOUD METADATA ENDPOINT (169.254.169.254):
 *   - Cloud environments mein yeh COMMONLY USED hai as a METADATA ENDPOINT
 *   - Instances aur workloads ke liye
 *   - Yeh expose kar sakta hai:
 *     * Credentials
 *     * Tokens
 *     * Configuration
 *   - Platform pe depend karta hai kya milega
 *
 * SECURITY WARNING:
 *   - Metadata endpoints ki access ko SECURITY-SENSITIVE treat karo
 *   - Especially un systems mein jo USER-PROVIDED URLs fetch karte hain
 *   - Yeh bahut dangerous ho sakta hai agar koi attacker isse access kar le
 *
 * ============================================================================
 *  SECTION 9: HOW ROUTING WORKS
 * ============================================================================
 *
 * Jab ek packet ke paas DESTINATION IP hota hai:
 *   - Routers decide karte hain ki use AAGE kahan bhejna hai (next hop)
 *
 * ============================================================================
 *  ROUTING TABLES
 * ============================================================================
 *
 * Routing table destination address ranges ko NEXT HOPS se map karta hai
 * Ek simplified routing table aisa dikhta hai:
 *
 *   Destination          Next Hop
 *   10.0.1.0/24          local subnet
 *   10.0.0.0/8           10.0.1.1
 *   0.0.0.0/0            internet gateway
 *
 * LONGEST PREFIX MATCH (Routers kaise choose karte hain):
 *   - Routers LONGEST PREFIX MATCH use karte hain
 *   - Matlab: SABSE SPECIFIC matching route JEETTA hai
 *
 *   Example:
 *     Packet for 10.0.1.25 matches BOTH:
 *       10.0.0.0/8  AND  10.0.1.0/24
 *     /24 JEETTA hai kyunki yeh CHHOTA aur ZYADA SPECIFIC range describe
 *     karta hai
 *
 * ============================================================================
 *  HOP-BY-HOP FORWARDING
 * ============================================================================
 *
 * - IP routing HOP BY HOP hai
 * - Ek router ko POORI PATH pata honi ki zaroorat NAHI hai
 *   - Sirf BEST MATCHING ROUTE ke liye NEXT HOP pata hona chahiye
 *
 * - Har IP packet mein kam se kam 2 ADDRESSES hote hain:
 *     * SOURCE IP: Packet KAHAN SE aaya dikh raha hai
 *     * DESTINATION IP: Packet kahan DELIVER hona chahiye
 *
 * - Routers kya karte hain:
 *     1. Destination IP dekhte hain
 *     2. Apni routing table se COMPARE karte hain
 *     3. Packet ko NEXT HOP pe forward karte hain
 *
 * - Har router ko sirf NEXT STEP pata hona chahiye
 *   - Poora path client se server tak jaanne ki zaroorat NAHI
 *
 * PATH INSPECT karne ke TOOLS:
 *   - traceroute
 *   - tracepath
 *
 * IMPORTANT - In tools ka output INCOMPLETE ya MISLEADING ho sakta hai:
 *   - Firewalls ke wajah se
 *   - ICMP filtering ke wajah se
 *   - Asymmetric routing ke wajah se
 *   - Load balancing ke wajah se
 *
 * Example traceroute output (documentation IP ranges):
 *   $ traceroute api.example.com
 *   1  192.168.1.1   1.2 ms  local gateway
 *   2  10.0.0.1      5.8 ms  ISP gateway
 *   3  198.51.100.10 8.3 ms  provider network
 *   4  203.0.113.20  9.1 ms  edge network
 *   5  192.0.2.34    9.3 ms  destination
 *
 * NOTE: Yeh example DOCUMENTATION IP RANGES use karta hai
 *   - Real traceroute output mein addresses honge jo AAPKE ISP, cloud provider,
 *     CDN, ya transit networks ke hain
 *
 * ============================================================================
 *  TTL AND HOP LIMIT (Packets Ko Loop Hone Se Rokna)
 * ============================================================================
 *
 * - IPv4 ek TTL (Time to Live) field use karta hai
 * - IPv6 ek HOP LIMIT field use karta hai
 * - Dono ka PURPOSE same hai: PACKETS KO FOREVER LOOP HONE SE ROKNA
 *
 * KAISE KAAM KARTA HAI:
 *   - Har router value ko 1 se DECREMENT karta hai (ghatata hai)
 *   - Jab value 0 ho jaati hai:
 *     * Router packet ko DROP kar deta hai
 *     * Usually ek ICMP TIME EXCEEDED message bhejta hai
 *
 * TRACEROUTE KAISE KAAM KARTA HAI:
 *   - Yeh INCREASING TTL ya HOP LIMIT values ke saath packets bhejta hai
 *   - Phir un RUTERS ko record karta hai jo EXPIRATION report karte hain
 *   - Isse intermediate hops discover hote hain
 *
 * TTL ka flow:
 *   Source (TTL=64) -> Decrement -> Router 1 (TTL=63) -> Decrement ->
 *   Router 2 (TTL=62) -> Decrement -> Router 3 (TTL=61) -> Deliver ->
 *   Destination (TTL=60)
 *
 * ============================================================================
 *  BGP (Border Gateway Protocol - Internet Level Routing)
 * ============================================================================
 *
 * - Internet SCALE pe routing BGP se coordinate hota hai
 * - BGP ka full form: "Border Gateway Protocol"
 *
 * - Internet bahut saare LARGE NETWORKS se bana hai:
 *     * ISPs
 *     * Cloud providers
 *     * CDNs
 *     * Enterprises
 *     * Others
 *
 * - BGP un networks ko allow karta hai:
 *     1. ANNOUNCE karna ki woh KAUNSE IP RANGES tak pahunch sakte hain
 *     2. ROUTING POLICY ke according PATHS choose karna
 *
 * BGP POWERFUL hai lekin BLUNT hai:
 *   - Yeh NAHI jaanta ki aapka application HEALTHY hai ya nahi
 *   - Sirf itna jaanta hai ki ek IP range EK PATH ke through REACHABLE dikh raha
 *     hai
 *   - Yeh problems ho sakti hain:
 *     * Route leaks
 *     * Bad announcements
 *     * Accidental withdrawals
 *   - Inse BADE SERVICES UNREACHABLE ho sakte hain
 *
 * REAL EXAMPLE - October 2021 Facebook Outage:
 *   - Route changes ne Facebook, Instagram, aur WhatsApp ko GHANTO KE LIYE
 *     UNREACHABLE bana diya
 *   - Kyunki relevant network prefixes GLOBAL ROUTING se GAYAB ho gaye
 *   - Yeh HTTP problem NAHI tha
 *   - Yeh database problem NAHI tha
 *   - Yeh ROUTING LAYER PE REACHABILITY problem tha
 *
 * ============================================================================
 *  MAC vs IP ADDRESSES (Local vs Cross-Network)
 * ============================================================================
 *
 * MAC ADDRESSES:
 *   - Current network link pe LOCAL DELIVERY ke liye use hote hain
 *   - Jaise aapke laptop se aapke router tak
 *
 * IP ADDRESSES:
 *   - Networks ke across ROUTING ke liye use hote hain
 *
 * IMPORTANT DIFFERENCE:
 *   - Jab traffic aapka local network chhodta hai:
 *     * DESTINATION MAC ADDRESS har hop pe CHANGE hota hai
 *     * DESTINATION IP usually SAME rehta hai
 *     * Jab tak packet target tak na pahunche
 *     * Ya koi device ise NAT ya proxying se REWRITE na kare
 *
 * ============================================================================
 *  SECTION 10: IP ADDRESSES IN SYSTEM DESIGN
 * ============================================================================
 *
 * IP addressing choices asar daalte hain:
 *   - Reliability pe (kitna reliable hai system)
 *   - Security pe (kitna secure hai)
 *   - Day-to-day operations pe (roz ka kaam)
 *
 * ============================================================================
 *  DO NOT TREAT IP AS IDENTITY (IP Ko Identity Mat Samjho!)
 * ============================================================================
 *
 * IP address:
 *   - CHANGE ho sakta hai
 *   - SHARE ho sakta hai
 *   - TRANSLATE ho sakta hai
 *   - PROXY, LOAD BALANCER, ya NAT GATEWAY represent kar sakta hai
 *
 * KABHI BHI IP address ko AKELA:
 *   - Permanent user identity ke roop mein mat use karo
 *   - Security decision ke roop mein mat use karo
 *
 * COMMON PITFALLS (Galtiyan Jo Log Karte Hain):
 *   1. Rate limiting by public IP:
 *      - Ek NAT ke peeche BAHUT SAARE unrelated users ko GROUP kar sakta hai
 *      - Unrelated users ko punish karna galat hai
 *
 *   2. Allowlisting office IPs:
 *      - Jab employees VPNs, mobile networks, ya cloud-hosted tools use karte
 *        hain toh yeh BREAK ho jaata hai
 *
 *   3. Logs mein proxy ya load balancer IP dikhta hai:
 *      - Jab tak original client IP forwarding CORRECTLY CONFIGURED nahi hai
 *
 *   4. Containers aur pods ke paas SHORT-LIVED addresses hote hain:
 *      - Yeh frequently change hote hain
 *
 * BEST PRACTICE:
 *   - IPs ko EK SIGNAL ke roop mein use karo
 *   - PURI IDENTITY MODEL mat banao IPs se
 *
 * ============================================================================
 *  PRESERVE CLIENT ADDRESS CAREFULLY
 * ============================================================================
 *
 * Applications ko original client IP ki zaroorat hoti hai:
 *   - Abuse detection (galat use pakadna)
 *   - Geolocation (kahan se aa raha hai)
 *   - Audit logs (kaun aaya, kab aaya)
 *   - Rate limiting
 *
 * PROBLEM:
 *   - Jab traffic PROXIES ya LOAD BALANCERS se guzarta hai
 *   - Application ko jo source IP dikhta hai woh PROXY ka ho sakta hai
 *
 * COMMON MECHANISMS (Original IP Preserve Karne Ke Tarike):
 *   1. X-Forwarded-For (HTTP header)
 *   2. Forwarded (Standard HTTP header)
 *   3. Proxy Protocol (TCP level)
 *   4. Cloud load balancer metadata
 *
 * SECURITY WARNING:
 *   - SIRF apne infrastructure se TRUST karo
 *   - Ek CLIENT HTTP HEADERS FAKE kar sakta hai
 *   - Jab tak ek TRUSTED PROXY unhe OVERWRITE ya CLEAN na kare pehle
 *
 * ============================================================================
 *  PLAN FOR ADDRESS EXHAUSTION
 * ============================================================================
 *
 * Address exhaustion sirf PUBLIC POOL ka problem nahi hai:
 *   - PRIVATE NETWORKS mein bhi hota hai
 *   - Large cloud estates mein
 *   - Kubernetes clusters mein
 *   - Lab platforms mein
 *   - Multi-region networks mein
 *
 * WATCH FOR (Dhyan Rakho):
 *   1. Too-small VPC ranges
 *   2. Overlapping networks after mergers ya cloud migrations
 *   3. Kubernetes pod address exhaustion
 *   4. NAT port exhaustion
 *   5. IPv4-only dependencies in systems jo dono support karte hain
 *   6. Hidden assumptions in firewall rules aur allowlists
 *
 * JITNI JALDI ADDRESSING DESIGN FIX KAROGE, UTNA SASTA HOGA
 *
 * ============================================================================
 *  PREFER NAMES FOR SERVICE CONTRACTS (Dubara - Bahut Important)
 * ============================================================================
 *
 * Applications ko usually DEPEND karna chahiye:
 *   - DNS names pe
 *   - Service discovery names pe
 *
 * HARD-CODED IP ADDRESSES NAHI use karna chahiye:
 *   - IPs IMPLEMENTATION DETAILS hain
 *
 * NAMES se KYA FAYDA:
 *   1. Failover (backup pe switch)
 *   2. Migration (server move)
 *   3. Load balancing
 *   4. Certificate validation
 *   5. Regional routing
 *
 * HARD-CODED IPs kabhi kabhi appropriate hain:
 *   - Infrastructure controls ke liye
 *   - Lekin RARE hone chahiye
 *   - Aur DOCUMENTED hone chahiye
 *
 * ============================================================================
 *  SUMMARY (Poore Document Ka Saransh)
 * ============================================================================
 *
 * 1. IP addresses ROUTING ADDRESSES hain
 *    - Yeh batate hain ki packets network context ke andar kahan jaayein
 *    - Yeh PROVE NAHI karte ki user kaun hai ya device permanently kya hai
 *
 * 2. IPv4 abhi bhi BAHUT ZYADA use hota hai
 *    - Chahe public IPv4 space exhausted hai
 *    - CIDR ne CLASSFUL ADDRESSING ko replace kiya
 *    - CIDR modern subnet design ki BUNIYAD hai
 *
 * 3. Private IPv4 ranges REUSABLE hain
 *    - Lekin OVERLAPPING RANGES SERIOUS ROUTING PROBLEMS create karte hain
 *    - NAT IPv4 ko USABLE rakhta hai
 *    - Lekin NAT yeh add karta hai:
 *      * State
 *      * Failure modes
 *      * Logging complexity
 *      * Scaling limits
 *
 * 4. IPv6 WIDELY DEPLOYED hai
 *    - Yeh modern designs mein BELONG karta hai
 *    - Usually DUAL-STACK operation ke through
 *
 * 5. KUCH DETAILS ACHHI TARAH SE SEEKHNE LAYAK HAIN:
 *    - Special addresses jaise 127.0.0.1, 0.0.0.0, 169.254.169.254, aur
 *      0.0.0.0/0
 *    - Yeh alag alag contexts mein ALAG matlab rakhte hain
 *    - Routing LONGEST PREFIX MATCH aur HOP-BY-HOP FORWARDING se kaam karta hai
 *
 * 6. IP-based controls USEFUL hain
 *    - Lekin yeh COMPLETE identity ya security model NAHI hain
 *    - Ek SOLID addressing plan rarely noticed hota hai (koi notice nahi karta)
 *    - Ek WEAK addressing plan YEH PROBLEMS create karta hai:
 *      * Failed peering
 *      * Exhausted pod ranges
 *      * Broken VPNs
 *      * NAT bottlenecks
 *      * Bad logs
 *      * Services jo ek dusre tak NAHI pahunch sakti
 *
 * ============================================================================
 **/